# 05 — BAR-ATH: fusion, plausibility, candidate (aggregates only)

Full-data members (exp_3: 3 new fits; exp_2: cached), broadcast fusion with the pinned exp_0 rule and the replay guard, label-free plausibility aggregates (never a choice), the adoption tree's candidate CSV (official validator) and its byte-identical rebuild from independent fits. No public-window check.

In [1]:
from pathlib import Path
import os, sys, json, time
ROOT = Path.cwd()
if not (ROOT / 'exp3_common.py').exists():
    raise RuntimeError('Run with working directory exp_jeremy/exp_3')
sys.path.insert(0, str(ROOT))
sys.dont_write_bytecode = True
import numpy as np
import pandas as pd
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)
import exp3_common as C
import exp1_common as C1
protocol = C.verify_prereg()
print('PREREG verified', protocol['prereg_sha256'][:16], '| frozen', protocol['frozen_at_utc'])
import exp3_protocol as P, exp3_target as E
assert C.public_check_spent(), 'exp_1 public check marker missing'

PREREG verified cff444fff0106e9c | frozen 2026-11-08T16:53:50+00:00


In [2]:
t0 = time.time()
match, items, fusion = E.inputs()
C1.write_json(C.RESULTS / 'target_fusion.json', fusion)
print(json.dumps({k: v for k, v in fusion.items() if k != 'halves'}, indent=1))
for half in fusion['halves']:
    print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in half.items()})
print(f'{time.time() - t0:.0f}s')

{
 "accepted_both_halves": false,
 "use_fused_input": false,
 "recovered_before_guard": 0,
 "kept_after_guard": 0,
 "removed_by_replay_guard": 0,
 "missing_ball_frames_graded": 64643,
 "recovered_share_of_missing_graded": 0.0,
 "ball_coverage_graded_tactical": 0.593074274347369,
 "ball_coverage_graded_fused": 0.593074274347369
}
{'half': 1, 'hypothesized_offset_frames': 4014, 'overlap_frames': 38449, 'median_distance_m': 0.789, 'p90_distance_m': 6.711, 'accepted': False, 'recovered_frames': 0}
{'half': 2, 'hypothesized_offset_frames': 26920, 'overlap_frames': 35752, 'median_distance_m': 0.836, 'p90_distance_m': 6.314, 'accepted': False, 'recovered_frames': 0}
7s


In [3]:
t0 = time.time()
models = {arm: E.full_model(arm) for members in E.MEMBERS.values() for arm in members}
plausibility = E.plausibility(match, items, models)
C1.write_json(C.RESULTS / 'target_plausibility.json', plausibility)
keys = ('dead_share', 'possessions_per_min', 'mean_possession_s', 'possession_count', 'dead_share_minus_prior')
print(pd.DataFrame({k: {m: v.get(m) for m in keys} for k, v in plausibility.items() if isinstance(v, dict)}).T.round(3).to_string())
print('prior dead share', plausibility['published_dead_share_prior'], '| ryu possessions/min', plausibility['ryu_possessions_per_min'])
print(f'{time.time() - t0:.0f}s')

                          dead_share  possessions_per_min  mean_possession_s  possession_count  dead_share_minus_prior
exp1_setseq_shipped            0.384                4.126              8.957             437.0                  -0.023
exp2_candidate_exp2_aug        0.408                3.569              9.959             378.0                   0.001
exp3_ctrl_ens3__tactical       0.417                2.408             14.517             255.0                   0.010
exp3_ref_ens3__tactical        0.410                2.446             14.465             259.0                   0.003
prior dead share 0.407 | ryu possessions/min 2.97
187s


In [4]:
t0 = time.time()
summary = E.build_candidate(match, items, models, fusion)
summary['reproduction'] = E.reproduce(match, items, fusion)
C1.write_json(C.RESULTS / 'target_candidate.json', summary)
print(json.dumps({k: v for k, v in summary.items() if k != 'share_graded'}, indent=1))
print(f'{time.time() - t0:.0f}s')

{
 "variant": "exp3_ref_ens3",
 "members": [
  "exp2_aug",
  "exp2_aug_s1",
  "exp2_aug_s2"
 ],
 "input": "tactical",
 "sha256": "17eefbcc0833973ee7cf5c8cc57fb6a12685c620b88aee3516342c6648737c08",
 "format_valid": true,
 "decoder": {
  "temperature": 1.169894951392387,
  "lambda_team": 8.0,
  "lambda_dead": 4.0,
  "gamma": 0.0
 },
 "public_labels_used": false,
 "graded_frames": 158857,
 "possession_count": 259,
 "abstain_frames_graded": 0,
 "dead_share": 0.41040684389104665,
 "possessions_per_min": 2.445595724456587,
 "mean_possession_s": 14.465019305019304,
 "dead_share_minus_prior": 0.0034068438910466736,
 "possessions_per_min_minus_ryu": -0.5244042755434131,
 "reproduction": {
  "first_sha256": "17eefbcc0833973ee7cf5c8cc57fb6a12685c620b88aee3516342c6648737c08",
  "second_sha256": "17eefbcc0833973ee7cf5c8cc57fb6a12685c620b88aee3516342c6648737c08",
  "byte_identical": true,
  "independent_fits": 3
 }
}
182s
